# Photo Labeling Tool

Click a button under each photo to label it. Progress saves to `labels.csv` after every click, so you can close this and resume anytime, already-labeled photos are skipped automatically.

**Before running**: export your food photos from Photos app (File -> Export -> Export Unmodified Originals) into the `food_photos/` folder next to this notebook.

In [ ]:
import os
import csv
from pathlib import Path
from datetime import datetime

import pillow_heif
pillow_heif.register_heif_opener()  # lets PIL open .heic files, iPhone's default format

from PIL import Image, ExifTags
from IPython.display import display, clear_output

PHOTO_DIR = Path("food_photos")
LABELS_CSV = Path("labels.csv")

VALID_EXTENSIONS = {".jpg", ".jpeg", ".heic", ".png"}

In [ ]:
def get_exif_timestamp(image_path):
    """Pull the original capture timestamp from a photo's EXIF data, if present."""
    try:
        img = Image.open(image_path)
        exif = img.getexif()
        if not exif:
            return None
        for tag_id, value in exif.items():
            tag_name = ExifTags.TAGS.get(tag_id, tag_id)
            if tag_name == "DateTime":
                return datetime.strptime(value, "%Y:%m:%d %H:%M:%S")
        return None
    except Exception:
        return None


def load_existing_labels():
    """Read labels.csv if it exists, so we know which photos are already done."""
    if not LABELS_CSV.exists():
        return set()
    with open(LABELS_CSV, "r") as f:
        reader = csv.DictReader(f)
        return {row["filename"] for row in reader}


def append_label(filename, label, timestamp):
    """Add one row to labels.csv, creating the file with a header if needed."""
    file_exists = LABELS_CSV.exists()
    with open(LABELS_CSV, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["filename", "label", "timestamp"])
        writer.writerow([filename, label, timestamp.isoformat() if timestamp else ""])

In [ ]:
all_photos = sorted(
    p for p in PHOTO_DIR.glob("*") if p.suffix.lower() in VALID_EXTENSIONS
) if PHOTO_DIR.exists() else []

already_labeled = load_existing_labels()
remaining = [p for p in all_photos if p.name not in already_labeled]

print(f"Found {len(all_photos)} photos in {PHOTO_DIR}/")
print(f"Already labeled: {len(already_labeled)}")
print(f"Remaining to label: {len(remaining)}")

if not all_photos:
    print("\nNo photos found. Export your food photos into the food_photos/ folder first.")

In [ ]:
# The labeling loop: shows one photo at a time, you type a letter + Enter to label it.
# Uses plain IPython display() + input(), not ipywidgets, so it doesn't depend on
# any extension-side widget manager (which was failing to register widget modules).
#
# Controls: a = allergen, s = safe, k = skip, q = quit (progress is saved either way)

from IPython.display import display, clear_output

queue = list(remaining)

for photo_path in list(queue):
    clear_output(wait=True)
    img = Image.open(photo_path).convert("RGB")
    img.thumbnail((500, 500))
    display(img)
    print(f"{photo_path.name}  ({len(queue)} remaining)")

    response = input("Label this photo [a=allergen, s=safe, k=skip, q=quit]: ").strip().lower()

    if response == "q":
        print("Stopped early, progress saved so far.")
        break
    elif response == "a":
        label = "allergen"
    elif response == "s":
        label = "safe"
    elif response == "k":
        label = "skip"
    else:
        print(f"Didn't recognize '{response}', treating as skip.")
        label = "skip"

    timestamp = get_exif_timestamp(photo_path)
    append_label(photo_path.name, label, timestamp)
    queue.remove(photo_path)

clear_output(wait=True)
print(f"Session done. {len(remaining) - len(queue)} photos labeled this session.")
print("Re-run this cell anytime to continue with whatever's left unlabeled.")

## Check Progress

Run this cell anytime to see a summary of labels collected so far.

In [ ]:
import pandas as pd

if LABELS_CSV.exists():
    labels_df = pd.read_csv(LABELS_CSV)
    print(f"Total labeled: {len(labels_df)}")
    print(labels_df["label"].value_counts())
else:
    print("No labels yet.")